<a href="https://colab.research.google.com/github/zhangling297/MAT_CS_599_deepLearningClassPracitices/blob/main/MAT499_599_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Question 1**
We would like to fit a 2-hidden layer fully connected neural network to this data, using 128 neurons
and 64 neurons in the hidden layers, respectively. Using 5-fold cross-validation, find the training
loss, training error, validation loss, and validation error for this model architecture for each fold. You
should train the model for 200 epochs with a mini-batch size of 16. Use the adam optimizer for this
learning with the default learning rate.
Create a table that shows the metrics for each fold and their averages.
Comment on the average training loss/error and validation error/loss for this model.

**Task**: Model the sonar dataset from the UCI Machine Learning
Repository (https://archive.ics.uci.edu/dataset/151/connectionist+bench+sonar+mines+vs+rocks).
A csv file of this data is available on the GitHub for this course
(https://raw.githubusercontent.com/benjaminmlucas/MAT499/refs/heads/main/module_2/sonar.c
sv). This dataset provides 60 features for each sound and these can be used to determine whether
the sound was a rock or a mine.


**Requirements **Set a seed to 599 for the numpy.random,
random, and torch libraries.

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Load the Sonar data CSV file into a DataFrame
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/sonar.csv"
df_Sonar = pd.read_csv(url, header=None)
col_names = [f"feature_{i}" for i in range(1, 61)] + ["label"]
df_Sonar.columns = col_names # Assign column names
print(df_Sonar.head())


   feature_1  feature_2  feature_3  feature_4  feature_5  feature_6  \
0     0.0200     0.0371     0.0428     0.0207     0.0954     0.0986   
1     0.0453     0.0523     0.0843     0.0689     0.1183     0.2583   
2     0.0262     0.0582     0.1099     0.1083     0.0974     0.2280   
3     0.0100     0.0171     0.0623     0.0205     0.0205     0.0368   
4     0.0762     0.0666     0.0481     0.0394     0.0590     0.0649   

   feature_7  feature_8  feature_9  feature_10  ...  feature_52  feature_53  \
0     0.1539     0.1601     0.3109      0.2111  ...      0.0027      0.0065   
1     0.2156     0.3481     0.3337      0.2872  ...      0.0084      0.0089   
2     0.2431     0.3771     0.5598      0.6194  ...      0.0232      0.0166   
3     0.1098     0.1276     0.0598      0.1264  ...      0.0121      0.0036   
4     0.1209     0.2467     0.3564      0.4459  ...      0.0031      0.0054   

   feature_54  feature_55  feature_56  feature_57  feature_58  feature_59  \
0      0.0159      0.

**Q1 Data precrocessing & FCNN model built**

In [3]:
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Define the target column name (already assigned in df_Sonar.columns)
target_column = df_Sonar.columns[-1]

# Separate features (X) and target (y)
X = df_Sonar.drop(columns=[target_column])
y = df_Sonar[target_column]

# Label encode the target variable
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_encoded = np.array(y_encoded, dtype=np.int64)

# Display the first few rows of X and y
print("Features (X) head:")
display(X.head())
print("Original Target (y) head:")
display(y.head())
print("Encoded Target (y_encoded) head:")
display(y_encoded[:5])


Features (X) head:


,feature_1,feature_2,feature_3,feature_4,feature_5,feature_6,feature_7,feature_8,feature_9,feature_10,...,feature_51,feature_52,feature_53,feature_54,feature_55,feature_56,feature_57,feature_58,feature_59,feature_60
0,0.0200,0.0371,0.0428,0.0207,0.0954,0.0986,0.1539,0.1601,0.3109,0.2111,...,0.0232,0.0027,0.0065,0.0159,0.0072,0.0167,0.0180,0.0084,0.0090,0.0032
1,0.0453,0.0523,0.0843,0.0689,0.1183,0.2583,0.2156,0.3481,0.3337,0.2872,...,0.0125,0.0084,0.0089,0.0048,0.0094,0.0191,0.0140,0.0049,0.0052,0.0044
2,0.0262,0.0582,0.1099,0.1083,0.0974,0.2280,0.2431,0.3771,0.5598,0.6194,...,0.0033,0.0232,0.0166,0.0095,0.0180,0.0244,0.0316,0.0164,0.0095,0.0078
3,0.0100,0.0171,0.0623,0.0205,0.0205,0.0368,0.1098,0.1276,0.0598,0.1264,...,0.0241,0.0121,0.0036,0.0150,0.0085,0.0073,0.0050,0.0044,0.0040,0.0117
4,0.0762,0.0666,0.0481,0.0394,0.0590,0.0649,0.1209,0.2467,0.3564,0.4459,...,0.0156,0.0031,0.0054,0.0105,0.0110,0.0015,0.0072,0.0048,0.0107,0.0094


Original Target (y) head:


,label
0,R
1,R
2,R
3,R
4,R


Encoded Target (y_encoded) head:


array([1, 1, 1, 1, 1])

In [2]:
# Define a fully connected model in Torch. Write a class for a fully-connected NN with two hidden layers with 128 and 64 neurons.

class FCNN(nn.Module):
    def __init__(self, input_size):
        super(FCNN, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.ReLU = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1) # Output layer with 1 neuron for binary classification

    def forward(self, x):
        x = self.ReLU(self.fc1(x))
        x = self.ReLU(self.fc2(x))
        x = torch.sigmoid(self.fc3(x)) # Apply sigmoid for binary classification output
        return x


In [4]:
# Set up 5-fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=599)

num_epochs = 200
batch_size = 16
fold_results = []


**Q1 Comments on Losses**
The average training loss and average trainning error are zero indicating that the model fits training data well and has perfectly classified the training data. However, the large discrepancy between training and validation existes when examining the average validation loss is 1.0812 and the average validation error is 0.1396. This indicating the FCNN model performs worse on unseen data, which is an evident of overfitting. ThIS FCNN model is better on memoring training data than generalizes to new data.



**Question 2**
Now add dropout to the model for regularization with a rate of 0.3. Perform the same 5-fold cross-
validation as above and create the same results table. Did dropout improve the model in your
educated opinion? Why / why not?
Repeat this process with a weight decay of 0.001 (removing the dropout). Perform the same 5-fold
cross-validation as above and create the same results table. Did weight decay improve the model in
your educated opinion? Why / why not?
Did dropout or weight decay work better on this model? Why?

**Q2 FCNN with dropout**

In [10]:

from sklearn.metrics import accuracy_score


# Dropout model with dropout rate (0.3)
class FCNN_Dropout(nn.Module):
    def __init__(self, input_size):
        super(FCNN_Dropout, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc2(x)
        x = self.relu(x)
        x = self.dropout(x)

        x = self.fc3(x)
        x = self.sigmoid(x)
        return x


# Encode labelsfor the full dataset
# Based on the current kernel state, y is available.
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y) # Changed y_train to y as per previous cell's output

# Create torch tensor datasets
X_all = torch.tensor(X.values, dtype=torch.float32) # Changed X_train to X as per previous cell's output
y_all = torch.tensor(y_encoded, dtype=torch.float32).view(-1, 1)


# Function for 5-fold CV
def run_cv_q2(model_class, X_tensor, y_tensor, kf, epochs=200, batch_size=16, weight_decay=0.0):
    criterion = nn.BCELoss()
    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X_tensor), start=1):
        # Split current fold
        # X_fold_train = X_tensor[train_idx] # This uses the tensor directly, but scaling needs numpy
        # y_fold_train = y_tensor[train_idx]
        # X_fold_val = X_tensor[val_idx]
        # y_fold_val = y_tensor[val_idx]

        # Data scaling
        X_train_fold_np = X.iloc[train_idx].values.astype(np.float32)
        # Get numpy array from original X
        X_val_fold_np = X.iloc[val_idx].values.astype(np.float32)
        y_train_fold_np = y_encoded[train_idx]
        y_val_fold_np = y_encoded[val_idx]

        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_fold_np)
        X_val_scaled = scaler.transform(X_val_fold_np)

        X_fold_train = torch.tensor(X_train_scaled, dtype=torch.float32)
        y_fold_train = torch.tensor(y_train_fold_np, dtype=torch.float32).view(-1, 1)
        X_fold_val = torch.tensor(X_val_scaled, dtype=torch.float32)
        y_fold_val = torch.tensor(y_val_fold_np, dtype=torch.float32).view(-1, 1)


        # Build datasets/loaders
        train_dataset = TensorDataset(X_fold_train, y_fold_train)
        val_dataset = TensorDataset(X_fold_val, y_fold_val)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Initialize model for this fold
        model = model_class(input_size=X_tensor.shape[1])

        # Adam with default learning rate; weight decay only when specified
        optimizer = optim.Adam(model.parameters(), weight_decay=weight_decay)

        # Train model
        for epoch in range(epochs):
            model.train()
            for data, target in train_loader:
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

        # Evaluate training and validation performance after training
        model.eval()
        with torch.no_grad():
            # Training metrics
            train_output = model(X_fold_train)
            train_loss = criterion(train_output, y_fold_train).item()
            train_pred = (train_output >= 0.5).float()
            train_error = 1 - accuracy_score(y_fold_train.numpy(), train_pred.numpy())

            # Validation metrics
            val_output = model(X_fold_val)
            val_loss = criterion(val_output, y_fold_val).item()
            val_pred = (val_output >= 0.5).float()
            val_error = 1 - accuracy_score(y_fold_val.numpy(), val_pred.numpy())

        fold_results.append({
            "Fold": fold,
            "Training Loss": train_loss,
            "Training Error": train_error,
            "Validation Loss": val_loss,
            "Validation Error": val_error
        })

    # Convert to DataFrame
    results_df = pd.DataFrame(fold_results)

    # Add average row
    avg_row = pd.DataFrame([{
        "Fold": "Average",
        "Training Loss": results_df["Training Loss"].mean(),
        "Training Error": results_df["Training Error"].mean(),
        "Validation Loss": results_df["Validation Loss"].mean(),
        "Validation Error": results_df["Validation Error"].mean()
    }])

    results_df = pd.concat([results_df, avg_row], ignore_index=True)
    return results_df


# Part 1: Dropout = 0.3

dropout_results = run_cv_q2(
    model_class=FCNN_Dropout,
    X_tensor=X_all,
    y_tensor=y_all,
    kf=kf,
    epochs=200,
    batch_size=16,
    weight_decay=0.0
)

print("Results for FCNN with Dropout (rate = 0.3)")
print(dropout_results.round(4))



# Part 2: Weight Decay = 0.001, no dropout

weight_decay_results = run_cv_q2(
    model_class=FCNN,          # baseline model from Q1
    X_tensor=X_all,
    y_tensor=y_all,
    kf=kf,
    epochs=200,
    batch_size=16,
    weight_decay=0.001
)

print("\nResults for FCNN with Weight Decay (0.001)")
print(weight_decay_results.round(4))


# Model comparision
drop_avg = dropout_results[dropout_results["Fold"] == "Average"].iloc[0]
wd_avg = weight_decay_results[weight_decay_results["Fold"] == "Average"].iloc[0]

comparison_df = pd.DataFrame({
    "Model": ["Dropout (0.3)", "Weight Decay (0.001)"],
    "Avg Training Loss": [drop_avg["Training Loss"], wd_avg["Training Loss"]],
    "Avg Training Error": [drop_avg["Training Error"], wd_avg["Training Error"]],
    "Avg Validation Loss": [drop_avg["Validation Loss"], wd_avg["Validation Loss"]],
    "Avg Validation Error": [drop_avg["Validation Error"], wd_avg["Validation Error"]]
})

print("\nAverage Performance Comparison")
print(comparison_df.round(4))

Results for FCNN with Dropout (rate = 0.3)
      Fold  Training Loss  Training Error  Validation Loss  Validation Error
0        1         0.0000          0.0000           0.7236            0.1667
1        2         0.0000          0.0000           2.7777            0.0952
2        3         0.0000          0.0000           1.0322            0.1429
3        4         0.0001          0.0000           0.6199            0.0976
4        5         0.0167          0.0060           1.5047            0.1707
5  Average         0.0034          0.0012           1.3316            0.1346

Results for FCNN with Weight Decay (0.001)
      Fold  Training Loss  Training Error  Validation Loss  Validation Error
0        1         0.0014             0.0           0.4676            0.1429
1        2         0.0014             0.0           0.4030            0.1190
2        3         0.0014             0.0           0.3691            0.1190
3        4         0.0014             0.0           0.4872        

**Question 2**

**Adding Dropout at a rate of 0.3 does not improve the FCNN model performance compared with the baseline FCNN model (without dropout)** After performing the same 5-fold cross-validation, results table shows that FCNN with dropout still yields the same training performance with the average training loss and training error valued zero. This indicate dropout did not change the FCNN model's performance on training data. On the validation dataset, dropout performs worse: the model average validation loss increases from 1.13 to 1.84 and the average validation error increases from 0.1331 to 0.1396. The reasons mignt be dropout wenaken model useful siginal learning, and may have introduced unnecessary randomness and noisy which hurt its performance on unseen data.

 **Compared with baseline model (FCNN Only),Weight decay slightly rises average training loss while training error stays at the same (Zero). Looking at validation dataset, weigh decay improves model performance** (removing the dropout and performing the same 5-fold cross-validation), weight decay significantly reduced the model average validation loss from 1.0812 to 0.4587 and the validation error drops from 0.1396 to 0.1298.. This inidcate the weight decay helps the model generalize better unseen data by mitigating overfitting. The reason might be weight decay adds penalties to the loss function and that encourages the model to learning more distributed weight values.

**Weight decay works better on this model** than dropout, because weight decay functions as a regularization method for the baseline FCNN and it controlled overfitting without harming validaitn performance. In contrast, dropout increased validation loss and validation error, which made the FCNN's performance on unseen data slightly worse, no improving on generalization.

**Question 3**
Using the preferred model from question 2, experiment with the learning rate. Try at least 5 values
and justify which is best for this problem and dataset.

In [9]:
#Setting: using FCNN with weight decay = 0.001 to test learning rates of 0.00001, 0,0001, 0.001, 0.01, 0.1) under the same 5-fold cross-validation

#Define FCNN model
class FCNNWeightDecay(nn.Module):
    def __init__(self, input_size):
        super(FCNNWeightDecay, self).__init__()
        self.fc1 = nn.Linear(input_size, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.sigmoid(x)
        return x

# Settings
kf = KFold(n_splits=5, shuffle=True, random_state=599)
batch_size = 32
num_epochs = 50
weight_decay_value = 0.001

# Apply to 5 learning rates
learning_rates = [0.00001, 0.0001, 0.001, 0.01, 0.1]

all_lr_results = []

for lr in learning_rates:
    print(f"\n==============================")
    print(f"Testing learning rate = {lr}")
    print(f"==============================")

    fold_results = []

    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        print(f'------------- Fold {fold+1}/{kf.n_splits} -------------')

        # Split fold
        X_train_fold, X_val_fold = X.iloc[train_idx], X.iloc[val_idx]
        y_train_fold, y_val_fold = y_encoded[train_idx], y_encoded[val_idx]

        # Scale using training fold only
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_fold).astype(np.float32)
        X_val_scaled = scaler.transform(X_val_fold).astype(np.float32)

        # Convert to torch tensors
        X_train_torch = torch.tensor(X_train_scaled, dtype=torch.float32)
        X_val_torch = torch.tensor(X_val_scaled, dtype=torch.float32)
        y_train_torch = torch.tensor(y_train_fold, dtype=torch.float32).view(-1, 1)
        y_val_torch = torch.tensor(y_val_fold, dtype=torch.float32).view(-1, 1)

        # DataLoaders
        train_dataset = TensorDataset(X_train_torch, y_train_torch)
        val_dataset = TensorDataset(X_val_torch, y_val_torch)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        # Initialize model
        model = FCNNWeightDecay(input_size=X_train_scaled.shape[1])

        # Loss and optimizer
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay_value)

        # Store epoch metrics
        fold_train_losses = []
        fold_train_errors = []
        fold_val_losses = []
        fold_val_errors = []

        for epoch in range(num_epochs):
            # Training
            model.train()
            current_train_loss = 0.0
            correct_train = 0
            total_train = 0

            for data, target in train_loader:
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()

                current_train_loss += loss.item() * data.size(0)
                predicted = (output > 0.5).float()
                total_train += target.size(0)
                correct_train += (predicted == target).sum().item()

            avg_train_loss = current_train_loss / len(train_loader.dataset)
            avg_train_error = 1 - (correct_train / total_train)

            # Validation
            model.eval()
            current_val_loss = 0.0
            correct_val = 0
            total_val = 0

            with torch.no_grad():
                for data, target in val_loader:
                    output = model(data)
                    loss = criterion(output, target)
                    current_val_loss += loss.item() * data.size(0)

                    predicted = (output > 0.5).float()
                    total_val += target.size(0)
                    correct_val += (predicted == target).sum().item()

            avg_val_loss = current_val_loss / len(val_loader.dataset)
            avg_val_error = 1 - (correct_val / total_val)

            fold_train_losses.append(avg_train_loss)
            fold_train_errors.append(avg_train_error)
            fold_val_losses.append(avg_val_loss)
            fold_val_errors.append(avg_val_error)

        # Save final epoch results for this fold
        fold_results.append({
            'Learning Rate': lr,
            'Fold': fold + 1,
            'Train Loss': fold_train_losses[-1],
            'Train Error': fold_train_errors[-1],
            'Validation Loss': fold_val_losses[-1],
            'Validation Error': fold_val_errors[-1]
        })

    # Convert fold results to DataFrame
    fold_df = pd.DataFrame(fold_results)

    # Compute average for this learning rate
    avg_row = pd.DataFrame({
        'Learning Rate': [lr],
        'Fold': ['Average'],
        'Train Loss': [fold_df['Train Loss'].mean()],
        'Train Error': [fold_df['Train Error'].mean()],
        'Validation Loss': [fold_df['Validation Loss'].mean()],
        'Validation Error': [fold_df['Validation Error'].mean()]
    })

    final_lr_df = pd.concat([fold_df, avg_row], ignore_index=True)
    print(final_lr_df.to_string(index=False))

    all_lr_results.append({
        'Learning Rate': lr,
        'Average Train Loss': fold_df['Train Loss'].mean(),
        'Average Train Error': fold_df['Train Error'].mean(),
        'Average Validation Loss': fold_df['Validation Loss'].mean(),
        'Average Validation Error': fold_df['Validation Error'].mean()
    })

# Final comparison table across learning rates
summary_df = pd.DataFrame(all_lr_results)

print("\n========================================")
print("Summary Across Learning Rates")
print("========================================")
print(summary_df.to_string(index=False))

# Choose the best learning rate based on lowest validation error,
# and use validation loss as secondary support
best_idx = summary_df['Average Validation Error'].idxmin()
best_lr_row = summary_df.loc[best_idx]

print("\nBest learning rate based on lowest average validation error:")
print(best_lr_row.to_string())
learning_rates = [0.00001, 0.0001, 0.001, 0.01, 0.1]
all_lr_results = []



Testing learning rate = 1e-05
------------- Fold 1/5 -------------
------------- Fold 2/5 -------------
------------- Fold 3/5 -------------
------------- Fold 4/5 -------------
------------- Fold 5/5 -------------
 Learning Rate    Fold  Train Loss  Train Error  Validation Loss  Validation Error
       0.00001       1    0.673886     0.265060         0.679279          0.357143
       0.00001       2    0.672426     0.445783         0.660290          0.357143
       0.00001       3    0.664900     0.457831         0.687070          0.500000
       0.00001       4    0.669404     0.395210         0.670673          0.341463
       0.00001       5    0.670968     0.329341         0.682085          0.390244
       0.00001 Average    0.670317     0.378645         0.675879          0.389199

Testing learning rate = 0.0001
------------- Fold 1/5 -------------
------------- Fold 2/5 -------------
------------- Fold 3/5 -------------
------------- Fold 4/5 -------------
------------- Fold 5/5 

**0.01 was the best learning rate among 0.00001, 0.0001, 0.001, 0.1** because it produces the lowest average validation error (0.12044), indicating the best generalization performance on unseen data. Smaller learning rates such as 0.0001 and 0.001 appears to learn too slowly, while larger learning rates such as 0.1 is too aggressive. Therefore, 0.01 is the best learning rate for this model and dataset.

**Question 4**

**Using cross-validation to justify all hyperparameter choices is a better approach than a single train-validation split?** because it evaluates the model on multiple different validation subsets rather than only one. Using single split may cause the estimated performance depend too much on which smaples happened to fall into the training set and which fell into the validation set. To contrast, using 5-fold cross-validation gives a more reliable estimate of generatlizaiton performance because every observation is used for validation once and for training multiple times. This makes hyperparameter choices less dependent on chance.

 **Using cross-validation to choose hyperparameters is especially important on small dataset** because it is small, so the validation results can vary noticeably depending on how the data is divided. The standard deviation of the validation error across the 5 folds helps show this. If the standard deviation is small, the model’s performance is relatively stable across different folds, meaning it generalizes more consistently. If the standard deviation is large, the validation performance changes more from fold to fold, showing that the model is sensitive to the particular split of the data. That would mean a single train-validation split could be misleading, since one split might look much better or worse than the model’s true average performance. Therefore, on this dataset, cross-validation is more important because it gives a more stable and dependable basis for choosing hyperparameters.